# Overfitting Detector Lab

*Companion to* **Mastering CatBoost — Pro Edition**, Chapter 4: Tuning & Diagnostics as System Control, §4.4 (The Autopilot: OD as a Safety System).

Guessing the "right" number of trees is rarely possible before training, because the optimal count depends on the learning rate and the data's noise level. CatBoost's answer is the *Iteration Overcommit* strategy: ask for far more trees than you need, monitor a held-out metric, and let the Overfitting Detector abort. This lab runs the detector's two modes (`Iter` and `IncToDec`) on deliberately noisy data, exposes both failure modes (premature stop and never-stops), and verifies exactly what `use_best_model` does to the final model.

In [ ]:
%pip install -q catboost==1.2.10 scikit-learn==1.6.1 numpy==2.0.2 pandas==2.2.2 matplotlib==3.10.0

import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier, Pool

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

## 1. A dataset built to overfit

Label noise (`flip_y=0.25`) plus a modest sample size gives the model plenty of noise to memorize — validation loss will genuinely turn upward, so the detector has something real to detect.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=6_000, n_features=20, n_informative=8,
                           n_redundant=4, flip_y=0.25, random_state=RANDOM_SEED)
X = X.astype(np.float32)
X_tr, X_rest, y_tr, y_rest = train_test_split(X, y, test_size=0.4, random_state=RANDOM_SEED, stratify=y)
X_va, X_te, y_va, y_te = train_test_split(X_rest, y_rest, test_size=0.5, random_state=RANDOM_SEED, stratify=y_rest)
print(f"train {X_tr.shape} | valid {X_va.shape} | test {X_te.shape}")

## 2. Four detector configurations, one dataset

Per §4.4.1:

- **`Iter` (patience mode)**: stop after `od_wait` consecutive iterations without a new best. In practice most users reach it through **`early_stopping_rounds`**, which is a convenience alias for `od_wait` and *forces* `od_type='Iter'` (any `od_pval` is discarded).
- **`IncToDec` (threshold mode)**: a statistical test on whether validation loss is genuinely rising; terminates when the p-value drops below `od_pval`. Statistically-motivated stop via od_pval; neither variant is categorically more aggressive -- behaviour depends on od_pval, patience, and the observed curve (see §4.4.1).

We compare: (a) no early stopping at all, (b) tight patience, (c) generous patience, (d) `IncToDec`.

In [ ]:
ITERS = 1500

def run_od(name, fit_kwargs=None, ctor_kwargs=None):
    ctor = dict(iterations=ITERS, depth=6, learning_rate=0.1, eval_metric="Logloss",
                random_seed=RANDOM_SEED, verbose=0, allow_writing_files=False)
    ctor.update(ctor_kwargs or {})
    m = CatBoostClassifier(**ctor)
    kw = dict(eval_set=(X_va, y_va), use_best_model=True)
    kw.update(fit_kwargs or {})
    t0 = time.perf_counter()
    m.fit(X_tr, y_tr, **kw)
    dt = time.perf_counter() - t0
    val_auc = roc_auc_score(y_va, m.predict_proba(X_va)[:, 1])
    return m, {"config": name,
               "stopped_after": len(m.get_evals_result()["validation"][ctor["eval_metric"]]),
               "best_iteration": m.get_best_iteration(),
               "final_trees": m.tree_count_,
               "validation_AUC": round(val_auc, 4),
               "fit_time_s": round(dt, 1)}

models, rows = {}, []
for name, fk, ck in [
    ("no early stopping",              {}, {}),
    ("Iter, od_wait=20 (tight)",       {"early_stopping_rounds": 20}, {}),
    ("Iter, od_wait=100 (generous)",   {"early_stopping_rounds": 100}, {}),
    ("IncToDec, od_pval=1e-2",         {}, {"od_type": "IncToDec", "od_pval": 1e-2}),
]:
    m, r = run_od(name, fk, ck)
    models[name] = m
    rows.append(r)
od_table = pd.DataFrame(rows)
display(od_table)

# Freeze the choice on validation evidence, then open the test set once.
champion_name = (od_table.sort_values(
    ["validation_AUC", "fit_time_s"], ascending=[False, True])
    .iloc[0]["config"])
champion = models[champion_name]
champion_test_auc = roc_auc_score(y_te, champion.predict_proba(X_te)[:, 1])
print(f"frozen champion: {champion_name}")
print(f"one-shot untouched test AUC: {champion_test_auc:.4f}")

Read your table against §4.4's mechanism figure: `stopped_after` is where training actually halted (best iteration + patience window for `Iter`), `best_iteration` is the red dot — the validation optimum — and `final_trees` shows that `use_best_model=True` truncated the model back to that optimum. Configuration comparison and the frozen champion choice use validation evidence only. The test set is opened once after that choice; its AUC is a final estimate, not another tuning signal.

In [ ]:
plt.figure(figsize=(9, 5))
colors = {"no early stopping": "gray", "Iter, od_wait=20 (tight)": "tab:orange",
          "Iter, od_wait=100 (generous)": "tab:blue", "IncToDec, od_pval=1e-2": "tab:green"}
for name, m in models.items():
    curve = m.get_evals_result()["validation"]["Logloss"]
    plt.plot(curve, label=f"{name} (stop={len(curve)})", color=colors[name],
             lw=1.5, alpha=0.9)
    plt.scatter([m.get_best_iteration()], [min(curve)], color=colors[name], zorder=5)
plt.xlabel("iteration"); plt.ylabel("validation Logloss")
plt.title("Overfitting Detector: where each configuration stopped (dots = best iteration)")
plt.legend(); plt.tight_layout(); plt.show()

## 3. What `use_best_model` actually does (§4.4.1)

The patience window means training always runs *past* the optimum. `use_best_model=True` then reverts the model to the best-iteration state; with `False` you keep the overfit tail. The book flags this as an "Important" verification step because defaults can vary with the training setup.

In [ ]:
for ubm in [True, False]:
    m = CatBoostClassifier(iterations=ITERS, depth=6, learning_rate=0.1,
                           random_seed=RANDOM_SEED, verbose=0, allow_writing_files=False)
    m.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=100,
          use_best_model=ubm)
    val_auc = roc_auc_score(y_va, m.predict_proba(X_va)[:, 1])
    print(f"use_best_model={ubm!s:5s} -> trees kept: {m.tree_count_:4d}, "
          f"best_iteration: {m.get_best_iteration()}, validation AUC: {val_auc:.4f}")

## 4. Label noise and the validation-selected stopping point

The sweep asks whether more planted label noise moves the validation minimum earlier in this seeded fixture. The direction is not universal: sampling, metric noise, regularization, and finite iteration ceilings can change the observed optimum. Report repeated-run results rather than treating the plot as a law.

In [ ]:
rows = []
for noise in [0.0, 0.1, 0.3]:
    Xn, yn = make_classification(n_samples=6_000, n_features=20, n_informative=8,
                                 n_redundant=4, flip_y=noise, random_state=RANDOM_SEED)
    Xn = Xn.astype(np.float32)
    Xn_tr, Xn_va, yn_tr, yn_va = train_test_split(Xn, yn, test_size=0.3,
                                                  random_state=RANDOM_SEED, stratify=yn)
    m = CatBoostClassifier(iterations=ITERS, depth=6, learning_rate=0.1,
                           random_seed=RANDOM_SEED, verbose=0, allow_writing_files=False)
    m.fit(Xn_tr, yn_tr, eval_set=(Xn_va, yn_va), early_stopping_rounds=100, use_best_model=True)
    rows.append({"label_noise": noise, "best_iteration": m.get_best_iteration(),
                 "stopped_after": len(m.get_evals_result()["validation"]["Logloss"])})
pd.DataFrame(rows)

## 5. Troubleshooting candidates

- If a run stops during an improving but noisy path, test larger patience, a different smooth evaluation metric, or a smaller step on development data.
- If it reaches the ceiling, inspect whether the curve is still improving, flat, or unstable before changing budget, capacity, features, or regularization.

## Takeaways

- Early stopping selects a prefix on development data; it does not prove the later trees are universally harmful.
- `Iter` and `IncToDec` are alternative detector rules. Neither is categorically aggressive or superior for a named dataset size.
- Patience, metric, learning rate, and ceiling interact; validate them under the actual compute and utility target.
- Label-noise effects on stopping are empirical.
- Freeze the choice before one-shot test evaluation.

## Exercises

**Exercise 1.** Compare no detector, `Iter`, and `IncToDec` on validation utility and cost under a predeclared rule. Report the selected candidate rather than prescribing one.

**Exercise 2.** Sweep learning rate and patience, repeat seeds, and report premature-stop diagnostics. Treat `N` roughly scaling with `1/eta` only as a budgeting heuristic.

**Exercise 3.** Compare stopping metrics and report the realized paths. Threshold metrics may be discontinuous, but AUC is rank-based rather than smooth in every finite update; choose from empirical stability and deployment relevance.

### Hidden solutions

<details><summary>Solution</summary>

**Exercise 1.** Fit no-detector, `Iter`, and `IncToDec` candidates on the same development splits. Capture validation best score, retained trees, ceiling reached, fit time, and score at a fixed common tree budget where available. Apply a predeclared utility/cost rule; a detector that stops sooner is not better unless its quality/cost trade-off satisfies that rule.

</details>

<details><summary>Solution</summary>

**Exercise 2.** Cross learning rates with patience values and repeat model seeds. Flag a premature stop only when a longer independently declared continuation improves validation by more than a tolerance. Tabulate false-stop rate, retained trees, best validation score, and time. Compare observed tree ratios with the `1/eta` budgeting heuristic instead of enforcing it.

</details>

<details><summary>Solution</summary>

**Exercise 3.** Train otherwise identical candidates monitored by Logloss and AUC, retain both full validation paths, and report how often their selected prefixes differ materially across seeds. Evaluate the frozen prefixes using the deployment metric. Choose the monitor from stability and utility evidence, not from a universal smoothness label.

</details>
